In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Parameters, State, and Memory

In the previous section, we organized models as trees of modules so frameworks
can find every parameter automatically. Until now we could let the framework
manage those tensors behind the scenes without asking what else a model holds or
how much memory training requires. At small scale, those details rarely bite. At
large scale, they dominate every engineering decision. Consider an
8-billion-parameter model such as the Llama 3 model from
that section. Storing its weights in 32-bit floating point
takes 32 GB, yet *training* it requires four times as much—128 GB—before
processing a single example: every parameter needs a matching gradient, and the
Adam optimizer (that section) maintains two additional tensors of the same
size for running averages of the gradients and squared gradients. That already
exceeds the memory of an 80 GB GPU.

Computing such budgets accurately requires knowing every tensor a model owns and
the role each one plays. Everyday workflows depend on the exact same inventory:
checkpoints must capture every persistent tensor the forward pass depends on,
GPU transfers must move them all together, and fine-tuning must specify cleanly
which tensors the optimizer may update.

Frameworks unify all of these operations through *registration*
(that section): each layer registers its tensors into the
module tree under structured names, allowing optimizers, checkpoint savers, and
device-transfer routines to walk that single tree. This section surveys that
state tree in three parts. First, we distinguish trainable *parameters* from
non-trainable *buffers*. Second, we account for training memory in bytes per
parameter and show how to count parameters without allocating their storage.
Third, we cover two recurring parameter patterns—weight tying and parameter
freezing—used throughout language modeling and transfer learning.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## The State Tree

### Access by Path

Our running example is a compact residual MLP: an input layer projecting 20
features to 64 hidden units, two 64-wide residual blocks, and a 10-unit output
layer. We redefine `ResidualBlock` from that section here so
the notebook stands on its own—each block adds the output of a two-layer *body*
directly to its input.

In [ ]:
class ResidualBlock(nnx.Module):
    def __init__(self, num_hiddens, *, rngs):
        self.body = nnx.Sequential(
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs), nnx.relu,
            nnx.Linear(num_hiddens, num_hiddens, rngs=rngs))

    def __call__(self, X):
        return X + self.body(X)

rngs = nnx.Rngs(d2l.get_key())
net = nnx.Sequential(nnx.Linear(20, 64, rngs=rngs),
                     ResidualBlock(64, rngs=rngs),
                     ResidualBlock(64, rngs=rngs),
                     nnx.Linear(64, 10, rngs=rngs))
X = jax.random.normal(rngs(), (2, 20))
net(X).shape

Parameters sit at the leaves of the module tree. The dotted path to any leaf
(the figure) mirrors how we index into it in Python, which is handy
for inspecting a single layer during debugging. Optimizers and checkpoint
writers, however, need *every* leaf at once, so each framework provides a
full-tree traversal yielding all parameters alongside their paths.

In NNX it is `nnx.state(net, nnx.Param)`, whose flat view yields each array
with its path. Parameters do not store their gradients: `nnx.grad` returns
them as a second state tree with the same paths.

This residual MLP contains 12 parameter tensors in total. Let's inspect the
total count along with the four tensors belonging to the first residual block:

In [ ]:
params = nnx.state(net, nnx.Param)
flat = list(params.flat_state())
len(flat), [(path, leaf.shape) for path, leaf in flat[2:6]]

A path tuple like `('layers', 1, 'body', 'layers', 0, 'kernel')` mirrors `net.layers[1].body.layers[0].kernel`. With only trainable parameters present, `nnx.state(net)` matches `nnx.state(net, nnx.Param)`:

In [ ]:
([path for path, _ in nnx.state(net).flat_state()]
 == [path for path, _ in params.flat_state()])

Every framework utility traverses this same state tree under these paths. The
equality above holds only because every tensor in our residual MLP happens to be
trainable—which is not true of all layers.

### Parameters and Buffers

Suppose a model standardizes its inputs using a feature-wise mean and standard
deviation computed once from the training set. Those statistics must be saved
inside checkpoints (without them, predictions at inference time would be wrong)
and moved onto the GPU alongside the weights—yet the optimizer must *never*
update them via gradient descent, because they are data statistics rather than
learned weights. Storing them as plain Python attributes fails, since module
traversals ignore plain attributes (that section). Batch
normalization (that section, [@Ioffe.Szegedy.2015]) relies on
the exact same category of state: running means and variances updated during
training forward passes and applied during inference.

Such persistent, non-trainable tensors are commonly called *buffers*, and every
framework lets us register them so checkpoints and device moves include them
while optimizers skip them. This yields a simple three-way rule for every tensor
inside a layer: register it as a *parameter* if the optimizer should update it,
register it as a *buffer* if checkpoints and device transfers need it without
gradients, and keep it as an ordinary local variable if it lives only for one
forward/backward pass (such as intermediate activations cached by
backpropagation in that section). Architectural constants like layer
widths remain plain Python attributes.

To illustrate, we build a `Standardizer` module that normalizes its inputs with
a precomputed per-feature mean and standard deviation before applying a linear
layer.

Subclassing `nnx.Variable` (here `class Buffer(nnx.Variable)`) distinguishes buffers from `nnx.Param`.

In [ ]:
class Buffer(nnx.Variable):
    pass

class Standardizer(nnx.Module):
    def __init__(self, mean, std, *, rngs):
        self.mean = Buffer(mean)
        self.std = Buffer(std)
        self.out = nnx.Linear(4, 2, rngs=rngs)

    def __call__(self, X):
        return self.out((X - self.mean) / self.std)

sample = jax.random.normal(rngs(), (100, 4)) * jnp.arange(1., 5.)
standardizer = Standardizer(sample.mean(0), sample.std(0), rngs=rngs)
[(path, value.shape) for path, value in nnx.state(standardizer).flat_state()]

Inspecting the trainable vs. non-trainable state confirms that checkpoints see
all four tensors while optimizers see only the linear layer's weight and bias:

In [ ]:
{'params': list(nnx.state(standardizer, nnx.Param)),
 'buffers': list(nnx.state(standardizer, Buffer))}

Registration governs more than checkpointing and optimization: whole-model operations such as dtype casting (that section) and GPU transfers (that section) traverse the exact same state tree, moving parameters and buffers alike while leaving unregistered plain attributes behind.

## Training Memory

### Bytes per Parameter

How much memory does it take to train our residual MLP? Focusing on model and
optimizer state alone, memory scales with three per-parameter components: the
parameter itself, its gradient, and any per-parameter optimizer state. Plain
minibatch SGD keeps no state, so in 32-bit floating point (fp32, 4 bytes per
number) it consumes 8 bytes per parameter. Adam adds two fp32 tensors per
parameter—exponential moving averages of the gradient and of the squared
gradient. Because nearly all large-scale training runs use Adam or a close
variant, that 16-byte figure is the practical baseline:

| Training state                       | Precision | Bytes per parameter |
|--------------------------------------|-----------|---------------------|
| Weights                              | fp32      | 4                   |
| Gradients                            | fp32      | 4                   |
| Adam's average of the gradients      | fp32      | 4                   |
| Adam's average of their squares      | fp32      | 4                   |
| Total                                |           | 16                  |

These 16 bytes per parameter represent a strict lower bound. They exclude
intermediate *activations* saved during the forward pass for backpropagation
(which grow with batch size and sequence length), temporary kernel workspaces,
and allocator fragmentation—all of which we measure directly on a GPU in
that section. For our residual MLP, that baseline comes out to:

In [ ]:
leaves = jax.tree_util.tree_leaves(nnx.state(net, nnx.Param))
n = sum(x.size for x in leaves)
weights = sum(x.size * x.dtype.itemsize for x in leaves)   # fp32: 4 bytes
grads, adam_state = weights, 2 * weights
print(f'{n} parameters: {(weights + grads + adam_state) / 2**20:.2f} MiB '
      'for weights + gradients + Adam state')

We can double-check that count by hand: the input layer contributes
$20 \times 64 + 64 = 1344$ parameters, each residual block contributes
$2 \times (64 \times 64 + 64) = 8320$, and the output layer adds
$64 \times 10 + 10 = 650$, totaling 18,634 parameters. Adam's optimizer state is
not just a theoretical entry in a table—it is real memory allocated tensor by
tensor on the first update, adding two full copies of the model:

In [ ]:
optimizer = nnx.Optimizer(net, optax.adam(1e-3), wrt=nnx.Param)
grads = nnx.grad(lambda model: model(X).sum())(net)
optimizer.update(net, grads)
moments = sum(x.size for x in jax.tree.leaves(optimizer.opt_state)
              if hasattr(x, 'ndim') and x.ndim > 0)
moments == 2 * n

For our tiny MLP, that baseline is a negligible 0.28 MiB, which is why memory
barely crossed our minds in earlier chapters. Scale the exact same arithmetic to
an 8-billion-parameter model, however, and you immediately hit 128 GB regardless
of batch size. You might hope that 16-bit *mixed-precision* training
(that section) cuts this static floor in half. Surprisingly, it does
not: mixed precision keeps fp32 master copies of the weights and fp32 Adam
moments so tiny updates do not vanish under rounding
[@Micikevicius.Narang.Alben.ea.2018], with the 16-bit copies sitting on top.

### Counting Without Allocating

Every parameter count above inspected tensors that had already been allocated.
For an 8B model, that gets the order backwards: checking whether 32 GB of
weights fit in RAM should not require allocating 32 GB first! Fortunately,
counting parameters requires only shapes and dtypes, both of which are
determined purely by constructor arguments. Consider the output projection of
Llama 3 8B, mapping a 4,096-dimensional hidden state to logits over 128,256
vocabulary *tokens*. Its weight matrix has
$4096 \times 128{,}256 = 525{,}336{,}576$ entries—2.1 GB in fp32. Below, we
instantiate and count this layer without allocating its storage.

`nnx.eval_shape` traces the constructor abstractly, returning `jax.ShapeDtypeStruct` leaves with shapes and dtypes but zero allocated arrays:

In [ ]:
key = d2l.get_key()   # drawn outside the abstract trace
head = nnx.eval_shape(lambda: nnx.Linear(
    4096, 128256, use_bias=False, rngs=nnx.Rngs(key)))
leaves = jax.tree.leaves(nnx.state(head, nnx.Param))
n_head = sum(x.size for x in leaves)
print(leaves, f'{n_head:,} params, {4 * n_head / 1e9:.1f} GB in fp32')

Running the same abstract inspection across every layer of an 8B model yields
its exact 8-billion parameter count—and the 128 GB training floor—without
allocating a single byte of tensor memory.

## Shared and Frozen Parameters

### Tied Parameters

A single parameter tensor can play multiple roles inside a network. The classic
example comes from language modeling (that section), where a
model reads tokens from a vocabulary $V$ and predicts the next token. Its input
*embedding* is a $|V| \times d$ lookup table whose $v^\textrm{th}$ row is the
$d$-dimensional representation of token $v$, while its output head projects a
$d$-dimensional hidden state back to $|V|$ token logits—another $|V| \times d$
table with the exact same geometry and semantic meaning. *Tying* both layers to
one shared weight matrix eliminates $|V| \times d$ parameters and often improves
generalization [@Press.Wolf.2017; @Inan.Khosravi.Socher.2017]: tokens that
behave interchangeably as inputs naturally deserve similar scores as outputs,
and weight tying enforces a single unified token geometry. The memory savings
can be dramatic—in GPT-2 small [@Radford.Wu.Child.ea.2019], the tied
$50257 \times 768$ table accounts for roughly 38.6 million of the model's 124
million parameters, nearly a third of the entire network.

![One weight matrix W, two call sites: the embedding looks up a row by token id, the output head multiplies by its transpose to produce logits. Tying means both point at the same tensor, so gradients from both uses sum into one place.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-weight-tying.svg)

the figure illustrates the two call sites. Let's verify the
mechanics on a miniature language model, `ToyLM`, with a 100-token vocabulary
and hidden width $d = 16$.

`nnx.Embed.attend(h)` projects hidden states using the transpose of `self.emb`'s embedding table directly:

In [ ]:
class ToyLM(nnx.Module):
    def __init__(self, vocab_size, d, tied=True, *, rngs):
        self.tied = tied
        self.emb = nnx.Embed(vocab_size, d, rngs=rngs)
        self.body = nnx.Linear(d, d, rngs=rngs)
        self.head = None if tied else nnx.Linear(
            d, vocab_size, use_bias=False, rngs=rngs)

    def __call__(self, tokens):
        h = nnx.relu(self.body(self.emb(tokens)))
        if self.tied:
            return self.emb.attend(h)   # the embedding table, used as the head
        return self.head(h)

lm = ToyLM(vocab_size=100, d=16, rngs=rngs)
tokens = jax.random.randint(rngs(), (2, 8), 0, 100)
[(path, value.shape)
 for path, value in nnx.state(lm, nnx.Param).flat_state()]

Does the shared matrix get counted once or twice? Let's compare the tied model
against an untied counterpart.

In [ ]:
untied = ToyLM(vocab_size=100, d=16, tied=False, rngs=rngs)
(sum(x.size for x in jax.tree.leaves(nnx.state(lm, nnx.Param))),
 sum(x.size for x in jax.tree.leaves(nnx.state(untied, nnx.Param))))

The tied model contains $1600 + 272 = 1872$ parameters (one embedding table plus
the body layer), whereas the untied version adds a second $100 \times 16$ table
for 3472 parameters in all.

What happens during backpropagation when a tensor is used twice? By the
multivariable chain rule, each call site contributes its own gradient term, and
both contributions sum into the gradient of the shared tensor. Writing
$\mathbf{W}_\textrm{emb}$ and $\mathbf{W}_\textrm{head}$ for the two tables of
the untied model initialized to the same values as the tied matrix $\mathbf{W}$,
the two models compute the exact same function and satisfy

$$\frac{\partial L}{\partial \mathbf{W}} =
\frac{\partial L}{\partial \mathbf{W}_\textrm{emb}} +
\frac{\partial L}{\partial \mathbf{W}_\textrm{head}}.$$

Let's verify this identity numerically by copying the tied weights into the
untied model and comparing gradients on the same batch:

In [ ]:
untied.emb.embedding[...] = lm.emb.embedding
untied.body.kernel[...] = lm.body.kernel
untied.body.bias[...] = lm.body.bias
untied.head.kernel[...] = lm.emb.embedding.T
g_tied = nnx.grad(lambda model: model(tokens).sum())(lm)
g_untied = nnx.grad(lambda model: model(tokens).sum())(untied)
jnp.allclose(g_tied['emb']['embedding'],
             g_untied['emb']['embedding'] + g_untied['head']['kernel'].T)

### Frozen Parameters

When fine-tuning a pretrained network on a new task (that section),
a common strategy is to keep the pretrained backbone fixed and train only a
fresh output head, dramatically cutting both data requirements and optimizer
memory. Locking parameters against gradient updates is called *freezing*. Let's
freeze every layer except the output head of a fresh residual MLP, leaving just
650 of its 18,634 parameters trainable.

In NNX, a `trainable` filter selects the output head (`layers[3]`) and is passed to both `nnx.grad` and `nnx.Optimizer(..., wrt=trainable)`:

In [ ]:
finetune = nnx.Sequential(nnx.Linear(20, 64, rngs=rngs),
                          ResidualBlock(64, rngs=rngs),
                          ResidualBlock(64, rngs=rngs),
                          nnx.Linear(64, 10, rngs=rngs))

trainable = nnx.All(nnx.Param, lambda path, _: path[:2] == ('layers', 3))

def count(state):
    return sum(x.size for x in jax.tree.leaves(state))

n_train = count(nnx.state(finetune, trainable))
(n_train, count(nnx.state(finetune, nnx.Param)))

Taking one Adam step should now update only the output head while leaving all
earlier layers untouched. Let's check every parameter before and after the step.

In [ ]:
head_opt = nnx.Optimizer(finetune, optax.adam(0.1), wrt=trainable)
before = nnx.to_pure_dict(nnx.state(finetune, nnx.Param))
grads = nnx.grad(lambda model: model(X).sum(),
                 argnums=nnx.DiffState(0, trainable))(finetune)
head_opt.update(finetune, grads)
stepped = nnx.to_pure_dict(nnx.state(finetune, nnx.Param))
jax.tree_util.tree_map(lambda a, b: bool((a != b).any()), before, stepped)

Only the output head's weight and bias changed, and the optimizer allocated
Adam moments for just those 650 trainable parameters.
Freezing comes with two subtle pitfalls, neither of which raises an error.
First, freezing a parameter does *not* reclaim optimizer state that has already
been allocated. If an Adam optimizer was constructed over all parameters and has
already stepped, it continues holding 8 bytes per parameter of running moments
even for layers frozen later—harmless for 18,634 parameters, but costly at
8-billion-parameter scale (as explored in the exercises).

The second pitfall is even easier to overlook: freezing blocks gradients, yet
some layer state mutates *without* gradients. In particular, batch normalization
(that section) updates its running mean and variance during the
*forward pass* whenever the layer runs in training mode, completely bypassing
the optimizer. Simply freezing a BatchNorm layer's trainable weights therefore
does not automatically lock its running statistics. Let's pass a shifted batch
through a frozen BatchNorm layer in training mode—with no optimizer step at
all—and check whether its running mean drifts.

In [ ]:
bn = nnx.BatchNorm(4, use_running_average=False, momentum=0.9,
                   use_fast_variance=False, rngs=rngs)
before = bn.mean[...]
bn(jax.random.normal(rngs(), (8, 4)) + 3)   # training mode
bool(jnp.allclose(bn.mean, before))

Every parameter we inspected in this section—tied, frozen, or averaged—began
from initial values supplied by the framework. In the next section, we examine
how those initial values are chosen and how to control initialization across the
entire module tree.

## Summary

A model's persistent state forms a tree of named tensors, and registration
determines which framework operations see each tensor: optimizers update
*parameters*, whereas checkpoints and device transfers also include *buffers*
(persistent tensors without gradients, such as running statistics), and
unregistered attributes are silently skipped. Training in fp32 with Adam
requires a baseline of 16 bytes per parameter before accounting for activations
(weights, gradients, and two Adam moments—128 GB for an 8B model), and abstract
shape inspection lets us compute such budgets without allocating memory. Weight
tying aliases one tensor across multiple roles so it is counted once and
receives the sum of gradients from all call sites; freezing halts gradient
updates but neither releases existing optimizer state nor automatically stops
forward-pass updates to BatchNorm running statistics.

## Exercises

1. [code] **Running statistics.** This problem needs only one fact about
   the batch normalization layer of that section, the update
   below. In training mode the layer normalizes with the minibatch mean
   $\hat{\boldsymbol{\mu}}_{\mathcal{B}}$ and variance, and each training
   step updates the running mean $\bar{\boldsymbol{\mu}}$ by

   $$\bar{\boldsymbol{\mu}} \leftarrow \rho\, \bar{\boldsymbol{\mu}} + (1 - \rho)\, \hat{\boldsymbol{\mu}}_{\mathcal{B}}.$$

    1. A fresh layer starts from $\bar{\boldsymbol{\mu}} = \mathbf{0}$.
       Predict $\bar{\boldsymbol{\mu}}$ after the layer has processed $k$
       batches in training mode, each with minibatch mean 3 in every
       feature. Check the prediction for $k = 1, 5, 20$ on a fresh layer
       over 4 features, without any optimizer step, and read off the value
       of $\rho$ the layer uses by default.
    1. Suppose you registered $\bar{\boldsymbol{\mu}}$ as a trainable
       parameter and removed the update above. What gradient does the
       training loss send to $\bar{\boldsymbol{\mu}}$? How do the training
       loss and the evaluation accuracy then compare with those of an
       unmodified model?
    1. Find a loss in $\bar{\boldsymbol{\mu}}$ alone whose gradient step
       with learning rate $\eta$ on each minibatch reproduces the removed
       update, and state the $\eta$ that matches $\rho$.

2. [code] **Memory footprint.** Write a helper that reports, for each
   top-level block of `net`, its number of state entries and its bytes
   under fp32 Adam training with every parameter trained.
    1. Which block dominates? At what output width would the output layer
       overtake one residual block?
    1. Insert a batch normalization layer over the 64 hidden features
       between the two residual blocks. Extend the helper so that it costs
       the layer's running statistics correctly, and report each block's
       bytes per entry. For which blocks is the share of the total bytes
       larger than the share of the total entries, and for which is it
       smaller? Explain both directions.

3. [code] **Tying and freezing.** Consider the tied `ToyLM` of this
   section.
    1. Leave the embedding table out of the trainable filter, for example
       with `nnx.All(nnx.Param, nnx.Not(nnx.PathContains('emb')))`, and
       pass the filter to `nnx.grad` and `nnx.Optimizer`. How many
       parameters are trained, and can the head still adapt? What does
       this show about the interaction between tying and freezing?
    1. Starting from the values of `lm`, build a model whose embedding stays
       fixed while its head adapts, and verify both properties after one
       update. How many parameters does it store and how many does it
       train, compared with a fully trainable `lm`?

4. **Fine-tuning memory.** A language model has 8 billion parameters. Its
   output layer is a $4096 \times 128{,}256$ matrix, not tied to the
   embedding. All state is fp32, and the optimizer is Adam.
    1. How many bytes do weights, gradients, and Adam's running averages take when
       every parameter is trained? How many 80 GB GPUs does that state
       alone fill?
    1. Freeze everything except the output layer, and build the optimizer
       over the output layer only. How many bytes does the persistent state
       take now, and does it fit on one 80 GB GPU?
    1. Suppose instead that an optimizer built over all parameters took
       one step before the freeze and is kept afterwards. How many bytes
       remain allocated, and does the state still fit?

[Discussions](https://d2l.discourse.group/t/36753)